# 第 5 周练习：了解我（RAG）

## 练习目标（理念）

从 **Google Drive**（可选）拉取个人相关文档，整理成 `person-knowledge` 知识库，再走第 5 周标准管线：

**加载 → 分块 → 本地嵌入 → Chroma → 检索增强生成 → Gradio 聊天**

聊天时只根据检索到的上下文回答「关于我」的问题；每条回复底部可附带 token 用量提示。

## 和本课第 5 周概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 文档加载 | `DirectoryLoader` + Drive 下载 / 抽取 |
| Chunking | `RecursiveCharacterTextSplitter` |
| Embedding | `HuggingFaceEmbeddings(all-MiniLM-L6-v2)` |
| Vector store | `Chroma` 持久化到 `me_vector_database` |
| RAG | Retriever(`k=RETRIEVE_K`) + OpenRouter LLM |
| UI | Gradio `ChatInterface`（流式 + token 展示）|

## 怎么跑

1. `.env` 配好 `OPEN_ROUTER_API_KEY`（聊天与可选结构化必需）
2. 若用 Drive：准备 `client_secret.json`，按下方开关运行下载格
3. 也可直接往 `person-knowledge/` 放 `.md` / `.txt`，跳过 Drive
4. 从上到下运行到最后一格 `demo.launch()`

## 1. 安装依赖

下一格用 `%pip` 安装 Google API 客户端与 `python-docx`（读 `.docx`）。若环境已装过可跳过。

In [ ]:
# ========== 安装：Google Auth / Drive API + python-docx ==========

# 笔记本魔法：在当前内核里 pip 安装下列包（包名字符串保持原样）
%pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client python-docx

## 设置路径、模型与 OpenRouter 客户端

下面定义知识库目录、分块超参、嵌入 / LLM 模型名，并尝试从环境变量创建 OpenRouter 兼容客户端。

In [1]:
# ========== 导入：路径、进度条、OpenAI、Drive、LangChain、Gradio ==========

# 标准库 os：读环境变量
import os
# 标准库 re：清洗文件名非法字符
import re
# Path：拼知识库 / 暂存 / 向量库路径
from pathlib import Path
# datetime：写错误日志时打时间戳
from datetime import datetime
# load_dotenv：把 .env 读进环境变量
from dotenv import load_dotenv
# tqdm：下载 / 构建知识库时显示进度条
from tqdm import tqdm
# OpenAI：经 base_url 调用 OpenRouter
from openai import OpenAI
# DocxDocument：从 .docx 抽段落文本
from docx import Document as DocxDocument
# Google OAuth 凭证与本地授权流
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
# Drive API：build 服务、HttpError、流式下载
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseDownload
# LangChain：加载 Markdown、分块、本地嵌入、Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
# gradio：聊天 UI
import gradio as gr

In [ ]:
# ========== 常量：目录 / 分块 / 模型 id / system 模板 / OpenRouter 客户端 ==========

# override=True：.env 覆盖已有同名环境变量
load_dotenv(override=True)

# 最终给人读的知识库目录（.md）
KNOWLEDGE_DIR = Path("person-knowledge")
# Drive 下载暂存目录（.txt / .docx）
STAGING_DIR = Path("person-knowledge-downloads")
# Chroma 持久化路径
DB_PATH = Path("me_vector_database")

# RAG 分块大小与重叠
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
# 检索时取 top-k 块数
RETRIEVE_K = 1
# 聊天用模型：可被环境变量覆盖；默认 model id 保持原样
LLM_MODEL = os.getenv("GET_TO_KNOW_ME_LLM", "openai/gpt-4.1-nano")
# 本地嵌入模型名
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
# 可选：把长原文整理成 Markdown 的模型
STRUCTURE_MODEL = os.getenv("GET_TO_KNOW_ME_STRUCTURE_MODEL", "openai/gpt-4o-mini")
# 结构化用的 system prompt（影响模型行为，正文不翻译）
STRUCTURE_SYSTEM_PROMPT = """You are a helpful editor. Rewrite the following raw text into clear Markdown for a \"Get to Know Me\" knowledge base. Use headings, bullets, short paragraphs. Preserve all facts; do not add or invent. Output only the Markdown."""

# OpenRouter 密钥；有则建客户端，无则后面聊天会提示未配置
api_key = os.getenv("OPEN_ROUTER_API_KEY")
openai = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key) if api_key else None
if api_key:
    # 只打印前缀，确认已加载
    print(f"OPEN_ROUTER_API_KEY:   {api_key[:12]}...")
else:
    print("OPEN_ROUTER_API_KEY not set — add to .env for chat and optional structuring")

## Google Drive 配置

开关与过滤关键字控制是否从 Drive 拉文件、拉多少、是否强制重新登录。下一格定义下载 / 抽取 / 构建知识库的函数。

In [3]:
# ========== Drive 开关、过滤规则、MIME 白名单、结构化超参 ==========

# True：启用从 Google Drive 拉取
USE_GOOGLE_DRIVE = True
# 要列文件的文件夹 id；"root" 表示「我的云端硬盘」根
GOOGLE_DRIVE_FOLDER_ID = "root"
# True：删掉 token.json，强制浏览器重新授权
FORCE_REAUTH = False
# True：只下载、不跑 build_knowledge
STOP_AFTER_DOWNLOAD = False
# 最多下载多少个文件
DRIVE_DOWNLOAD_LIMIT = 40 
# True：文件名需命中 FILTER_KEYWORDS 才进入知识库
FILTER_BY_NAME = True
# 文件名过滤关键字元组（小写匹配）
FILTER_KEYWORDS = ("cv", "cover letter", "profile")
# 错误追加写入的日志路径
DRIVE_ERROR_LOG = Path("drive_errors.log")

def _log_err(msg):
    # 以追加模式写一行「时间戳 + 消息」
    with open(DRIVE_ERROR_LOG, "a", encoding="utf-8") as f:
        f.write(f"{datetime.now().isoformat()} {msg}\n")

# 允许的 Drive MIME 类型（纯文本 / docx / Google Docs）
ALLOWED_MIME = ["text/plain", "application/vnd.openxmlformats-officedocument.wordprocessingml.document", "application/vnd.google-apps.document"]
# 导出 Google Docs 时用的目标 MIME（docx）
GOOGLE_DOCS_EXPORT = "application/vnd.openxmlformats-officedocument.wordprocessingml.document"
# OAuth 只读 Drive 范围
SCOPES_DRIVE = ["https://www.googleapis.com/auth/drive.readonly"]
# 结构化前再切一刀的块大小 / 重叠
STRUCTURE_CHUNK_CHARS, STRUCTURE_OVERLAP = 2000, 200
# 短于该长度的文件直接原样写入，不调结构化 LLM
SKIP_STRUCTURE_IF_SHORTER_THAN = 2500
# 单文件结构化块数上限，超限则整文件原样写出以省内存
MAX_CHUNKS_PER_FILE = 100 

In [4]:
# ========== Drive 鉴权 / 列举 / 下载 / 抽文本 / 结构化写 .md ==========

def _client_secret_path(root=None):
    # 在给定根目录（默认 cwd）找 client_secret.json 或 credentials.json
    r = Path(root) if root else Path.cwd()
    for n in ("client_secret.json", "credentials.json"):
        if (r / n).exists(): return r / n
    # 都没有时仍返回默认期望路径，供上层报错提示
    return r / "client_secret.json"

def build_drive_service():
    # 定位密钥文件；token 与密钥同目录
    secret = _client_secret_path()
    token_path = secret.parent / "token.json"
    creds = None
    # 若已有 token：尝试加载
    if token_path.exists():
        try: creds = Credentials.from_authorized_user_file(str(token_path), SCOPES_DRIVE)
        except Exception: token_path.unlink(missing_ok=True)
    # 无效则尝试 refresh，或走本地浏览器 OAuth
    if not creds or not creds.valid:
        if creds and creds.expired and getattr(creds, "refresh_token", None):
            try: creds.refresh(Request())
            except Exception: creds = None
        if not creds or not creds.valid:
            if not secret.exists(): raise FileNotFoundError(f"Put client_secret.json at {secret}")
            flow = InstalledAppFlow.from_client_secrets_file(str(secret), SCOPES_DRIVE)
            creds = flow.run_local_server(port=0)
        # 把新凭证写回 token.json，下次免登
        token_path.parent.mkdir(parents=True, exist_ok=True)
        token_path.write_text(creds.to_json(), encoding="utf-8")
    # 返回 Drive v3 服务客户端
    return build("drive", "v3", credentials=creds)

def list_drive_files(service, folder_id, limit=None):
    # 查询：未进回收站，且 MIME 在白名单
    q = "trashed = false and (" + " or ".join(f"mimeType='{m}'" for m in ALLOWED_MIME) + ")"
    # 父文件夹；空则用 root
    parent_id = (folder_id or "").strip() or "root"  # "root" = My Drive home (drive.google.com/drive/home)
    q += f" and '{parent_id}' in parents"
    files, token = [], None
    while True:
        # 本页最多 100；若有 limit 则按剩余额度收窄
        n = min(100, (limit - len(files)) if limit else 100)
        if limit and n <= 0: break
        r = service.files().list(q=q, spaces="drive", fields="nextPageToken, files(id, name, mimeType, modifiedTime)", orderBy="modifiedTime desc", pageToken=token or "", pageSize=n).execute()
        files.extend(r.get("files", []))
        if limit and len(files) >= limit: return files[:limit]
        token = r.get("nextPageToken")
        if not token: break
    return files

def download_drive_file(service, meta, out_dir):
    # 取文件 id 与 MIME，决定扩展名
    fid, mime = meta["id"], meta.get("mimeType", "")
    ext = ".docx" if "document" in mime or mime == "application/vnd.google-apps.document" else ".txt"
    # 清洗文件名，截断长度
    name = re.sub(r'[^\w\s\-.]', '_', Path(meta.get("name", fid)).stem)[:200] or "untitled"
    path = Path(out_dir) / f"{name}{ext}"
    path.parent.mkdir(parents=True, exist_ok=True)
    # Google Docs 用 export；其它用 get_media
    req = service.files().export_media(fileId=fid, mimeType=GOOGLE_DOCS_EXPORT) if mime == "application/vnd.google-apps.document" else service.files().get_media(fileId=fid)
    # 分块写入本地文件直到 done
    with open(path, "wb") as f:
        d = MediaIoBaseDownload(f, req)
        while not d.next_chunk()[1]: pass
    return path

def extract_text(path):
    # 统一抽纯文本：.txt 直接读；.docx 拼非空段落
    path = Path(path)
    if path.suffix.lower() == ".txt": return path.read_text(encoding="utf-8", errors="replace")
    if path.suffix.lower() == ".docx":
        doc = DocxDocument(str(path))
        return "\n".join(p.text.strip() for p in doc.paragraphs if p.text.strip())
    raise ValueError("Only .txt and .docx supported")


def get_structure_splitter():
    # 结构化阶段用的分块器（与 RAG 分块参数不同）
    return RecursiveCharacterTextSplitter(chunk_size=STRUCTURE_CHUNK_CHARS, chunk_overlap=STRUCTURE_OVERLAP, length_function=len)

def structure_chunk(chunk, model=STRUCTURE_MODEL, openai_client=None):
    # 无密钥则原样返回，不调用 API
    key = os.getenv("OPEN_ROUTER_API_KEY")
    if not key: return chunk
    if openai_client is None:
        openai_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)
    # system 定编辑规则，user 放原文块
    result = openai_client.chat.completions.create(model=model, messages=[{"role": "system", "content": STRUCTURE_SYSTEM_PROMPT}, {"role": "user", "content": chunk}])
    return (result.choices[0].message.content or chunk).strip()

def build_knowledge(staging_dir, knowledge_dir, use_llm=True, filter_by_name=True, filter_keywords=("cv", "cover letter", "profile")):
    # 确保输出目录存在
    knowledge_dir = Path(knowledge_dir)
    knowledge_dir.mkdir(parents=True, exist_ok=True)
    raw = []
    # 暂存区全部 txt + docx
    paths = sorted(Path(staging_dir).glob("*.txt")) + sorted(Path(staging_dir).glob("*.docx"))
    # 可选：按文件名关键字过滤
    if filter_by_name and filter_keywords:
        key = [k.lower() for k in filter_keywords]
        paths = [p for p in paths if any(k in p.name.lower() for k in key)]
        tqdm.write(f"Filtered to {len(paths)} file(s) containing: {', '.join(filter_keywords)}")
    # 抽文本进 raw 列表
    for p in tqdm(paths, desc="Loading files"):
        try:
            t = extract_text(p)
            if t.strip(): raw.append((p.name, t.strip()))
        except Exception as e:
            msg = f"extract {p}: {e}"; _log_err(msg); tqdm.write(f"Error: {msg}")

    # 若要用 LLM 结构化：先建客户端（注意：下一行原逻辑会把 openai_client 再置 None）
    openai_client = None
    if use_llm and os.getenv("OPEN_ROUTER_API_KEY"):
        openai_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPEN_ROUTER_API_KEY"))
    openai_client = None
    splitter = get_structure_splitter()
    for name, text in tqdm(raw, desc="Building knowledge"):
        out = knowledge_dir / f"{Path(name).stem}.md"
        try:
            # 足够长：切块后逐块 structure_chunk 写入，中间用分隔符
            if len(text) > SKIP_STRUCTURE_IF_SHORTER_THAN:
                chunks = splitter.split_text(text)
                chunks = [c for c in chunks if c and c.strip()]
                if len(chunks) > MAX_CHUNKS_PER_FILE:
                    tqdm.write(f"  [{Path(name).stem}] {len(chunks)} chunks > {MAX_CHUNKS_PER_FILE}; writing as-is to save RAM.")
                    out.write_text(text, encoding="utf-8")
                    continue
                tqdm.write(f"  [{Path(name).stem}] {len(chunks)} chunk(s):")
                for i, c in enumerate(chunks):
                    tqdm.write(f"  --- Chunk {i+1}/{len(chunks)} ({len(c)} chars) ---")
                    tqdm.write((c[:600] + "...") if len(c) > 600 else c)
                    tqdm.write("")
                # 逐块写盘，避免先拼超大字符串占内存
                with open(out, "w", encoding="utf-8") as f:
                    for i, c in enumerate(tqdm(chunks, desc=f"  write ({Path(name).stem})", leave=False)):
                        piece = structure_chunk(c, openai_client=openai_client)
                        f.write(piece)
                        if i < len(chunks) - 1:
                            f.write("\n\n--- CHUNK ---\n\n")
            else:
                # 短文：直接原样写 .md
                out.write_text(text, encoding="utf-8")
        except Exception as e:
            msg = f"write {out}: {e}"; _log_err(msg); tqdm.write(f"Error: {msg}")
    # 返回知识库里所有 .md 路径
    return list(knowledge_dir.glob("*.md"))

In [ ]:
# ========== 执行：按开关从 Drive 下载，再 build_knowledge ==========

if USE_GOOGLE_DRIVE:
    # 确保暂存目录存在
    STAGING_DIR.mkdir(parents=True, exist_ok=True)
    # 暂存区是否已有文件（有则跳过下载）
    staging_has_files = list(Path(STAGING_DIR).glob("*.txt")) or list(Path(STAGING_DIR).glob("*.docx"))
    if staging_has_files:
        print(f"person-knowledge-download already has {len(staging_has_files)} file(s); skipping download.")
        if not STOP_AFTER_DOWNLOAD:
            # 直接用已有下载物构建知识库
            build_knowledge(STAGING_DIR, KNOWLEDGE_DIR, use_llm=True, filter_by_name=FILTER_BY_NAME, filter_keywords=FILTER_KEYWORDS)
            print(f"Knowledge -> {KNOWLEDGE_DIR.resolve()}")
    else:
        # 需要重新授权时删 token
        if FORCE_REAUTH:
            _tp = _client_secret_path().parent / "token.json"
            if _tp.exists(): _tp.unlink(missing_ok=True); print("Cleared token.json — sign in again in browser.")
        # 建服务 → 列文件 → 逐个下载
        svc = build_drive_service()
        files = list_drive_files(svc, GOOGLE_DRIVE_FOLDER_ID, limit=DRIVE_DOWNLOAD_LIMIT)
        errors = []
        for m in tqdm(files, desc="Downloading from Drive"):
            try:
                download_drive_file(svc, m, STAGING_DIR)
            except Exception as e:
                msg = f"download {m.get('name', m.get('id'))}: {e}"; _log_err(msg); errors.append(msg); tqdm.write(f"Error: {msg}")
        if errors: print(f"{len(errors)} error(s) -> {DRIVE_ERROR_LOG.resolve()}")
        print(f"Downloaded {len(files) - len(errors)}/{len(files)} files -> {STAGING_DIR.resolve()}")
        if STOP_AFTER_DOWNLOAD:
            print("Stopped after download (STOP_AFTER_DOWNLOAD=True). Set False to run build_knowledge.")
        elif files:
            build_knowledge(STAGING_DIR, KNOWLEDGE_DIR, use_llm=True, filter_by_name=FILTER_BY_NAME, filter_keywords=FILTER_KEYWORDS)
            print(f"Knowledge -> {KNOWLEDGE_DIR.resolve()}")
else:
    print("Set USE_GOOGLE_DRIVE=True to fetch from Drive.")

---

## 2. 从 person-knowledge 加载文档

用 `DirectoryLoader` 读取知识库目录下的 `.md` / `.txt`，供后续分块与嵌入。

In [ ]:
# ========== 加载知识库文件夹中的 .md / .txt ==========

def load_docs_from_folder(folder_path):
    folder_path = Path(folder_path)
    # 目录不存在就创建，方便手工塞文件或跑 Drive 流程
    folder_path.mkdir(parents=True, exist_ok=True)  # create if missing so you can add files or run 1b
    documents = []
    # 分别按扩展名 glob 加载，再合并
    for ext in ["*.md", "*.txt"]:
        loader = DirectoryLoader(str(folder_path), glob=ext, loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
        documents.extend(loader.load())
    return documents

# 对 KNOWLEDGE_DIR 执行加载
docs = load_docs_from_folder(KNOWLEDGE_DIR)
print(f"Loaded {len(docs)} document(s) from {KNOWLEDGE_DIR}")
if not docs:
    # 空目录提示：告诉你绝对路径，方便排查
    print(f"  (Folder is empty. Add .md or .txt files there.)")
    print(f"  Full path: {KNOWLEDGE_DIR.resolve()}")
# 列出每篇 source 与字符数
for d in docs:
    print(f"  - {d.metadata.get('source', '?')} ({len(d.page_content)} chars)")

---

## 3. 文本分块（Chunking）

把长文档切成带重叠的小块，便于嵌入与检索时命中局部语义。

In [ ]:
# ========== 按 CHUNK_SIZE / CHUNK_OVERLAP 分块 ==========

# length_function=len：按 Python 字符数计长度
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, length_function=len)
# 对上一格的 docs 切块
chunks = text_splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")
# 有块时打印第一块前 300 字符作抽样
if chunks:
    print("Example:", chunks[0].page_content[:300] + "...")

---

## 4. 在 Chroma 中做向量嵌入

本地 HuggingFace 模型把每个 chunk 编成向量，写入 `DB_PATH` 持久化目录。

In [ ]:
# ========== 本地嵌入 + Chroma.from_documents 落盘 ==========

# 加载 all-MiniLM-L6-v2（首次可能下载权重）
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
# 用 chunks 建库并 persist 到 DB_PATH
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=str(DB_PATH))
print(f"Vector store at {DB_PATH} with {len(chunks)} chunks")

## 5. 检索 + 生成（RAG）

把向量库变成 Retriever：先取相关上下文，再交给 OpenRouter 上的 LLM（支持流式，并尽量收集 token 用量）。

In [13]:
# ========== Retriever + system 模板 + 流式问答（附 usage） ==========

# k 取自 RETRIEVE_K（本笔记本默认为 1）
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVE_K})
# 要求只根据 context 回答；模板正文保持原样
SYSTEM_TEMPLATE = """You answer questions about the user based ONLY on the context below. If not in context, say so. Do not make up information.
Context:
{context}
"""

def get_context(question):
    # 语义检索；把多块用分隔符拼成一段 context
    docs = retriever.invoke(question)
    return "\n\n---\n\n".join(d.page_content for d in docs), docs

def answer_with_stream(question, history):
    # 先取上下文与原始 docs（便于上层调试）
    context, retrieved_docs = get_context(question)
    # 未配置 OpenRouter：返回提示字符串，不调用 API
    if openai is None:
        return f"[OpenRouter not configured] Set OPEN_ROUTER_API_KEY in .env. Retrieved {len(retrieved_docs)} chunk(s).", {}, retrieved_docs
    # messages：system（含 context）+ 历史 + 当前 user
    messages = [{"role": "system", "content": SYSTEM_TEMPLATE.format(context=context)}, *[{"role": h["role"], "content": h["content"]} for h in history], {"role": "user", "content": question}]
    full, usage = "", {}
    # 流式拼接 delta.content；若 chunk 带 usage 则记下
    for chunk in openai.chat.completions.create(model=LLM_MODEL, messages=messages, stream=True):
        if chunk.choices and chunk.choices[0].delta.content:
            full += chunk.choices[0].delta.content
        if chunk.usage:
            usage = {"prompt_tokens": chunk.usage.prompt_tokens, "completion_tokens": chunk.usage.completion_tokens}
    # 流式没拿到 usage 时：再打一枪非流式补 usage（并覆盖 full）
    if not usage and full:
        r = openai.chat.completions.create(model=LLM_MODEL, messages=messages)
        full = r.choices[0].message.content or ""
        if r.usage:
            usage = {"prompt_tokens": r.usage.prompt_tokens, "completion_tokens": r.usage.completion_tokens}
    return full or "(no response)", usage, retrieved_docs

---

## 6. Gradio UI（流式结果 + token 展示）

聊天界面：把旧版 `(user, bot)` 历史转成 messages，调用 `answer_with_stream`，并在回复末尾附加 completion token 提示。

In [ ]:
# ========== Gradio ChatInterface：包装 history + 附加 token 行 ==========

def chat_with_tokens(message, history):
    # 空消息直接返回空串
    if not message or not message.strip():
        return ""
    # 把 Gradio 旧格式 history 转成 [{role, content}, ...]
    hist = []
    for h in history or []:
        if h[0]: hist.append({"role": "user", "content": h[0]})
        if h[1]: hist.append({"role": "assistant", "content": h[1]})
    # 调 RAG 流式回答
    answer, usage, _ = answer_with_stream(message.strip(), hist)
    # 有 usage 时在文末追加 Markdown 提示行
    if usage:
        answer += f"\n\n---\n*Tokens:completion={usage.get('completion_tokens', '?')}*"
    return answer

# 标题与描述字符串保持原样
demo = gr.ChatInterface(chat_with_tokens, title="Get to Know Me — RAG", description="Ask about me.")
# 启动 Gradio 服务
demo.launch()